In [1]:
import json, ast, re
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

## The three validators, plus a chooser

In [3]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    fmt = test_case["format"]
    if fmt == "json":
        return validate_json(response)
    elif fmt == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

``grade_syntax`` is the function the lesson’s last snippet calls. It picks the validator from the format field.

## Regenerate the dataset with a format field

In [4]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
print(dataset)

[{'task': "Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket-name/path/to/object')", 'format': 'regex'}, {'task': 'Create a JSON CloudFormation template snippet that defines an AWS Lambda function with basic execution role and environment variables', 'format': 'json'}, {'task': 'Write a Python function that takes an AWS IAM ARN string and extracts the account ID, resource type, and resource name', 'format': 'python'}]


Check that each item has ``"format"`` with exactly ``json``, ``python`` or ``regex``.

## Stricter prompt

In [5]:
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

The prefill and stop sequence strip the code fences, so the validators receive raw code.

## Simple model grader

In [6]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}

Solution:
{output}

Reply in JSON with keys "strengths" (list), "weaknesses" (list),
"reasoning" (string) and "score" (number 1 to 10).
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"], temperature=0.0)
    return json.loads(text)

## Combine the scores and run

In [8]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": model_grade["reasoning"],
    }

def run_eval(dataset):
    return [run_test_case(tc) for tc in dataset]

results = run_eval(dataset)
for r in results:
    print(r["score"], "-", r["test_case"]["task"])
print("Average:", sum(r["score"] for r in results) / len(results))

8.0 - Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket-name/path/to/object')
8.0 - Create a JSON CloudFormation template snippet that defines an AWS Lambda function with basic execution role and environment variables
7.5 - Write a Python function that takes an AWS IAM ARN string and extracts the account ID, resource type, and resource name
Average: 7.833333333333333
